# Model Experiment: Prophet

**ამოცანა:** Walmart Store Sales Forecasting — Classical Statistical Time-Series მოდელები

ამ notebook-ში განვიხილავთ **Prophet**-ს (Meta/Facebook-ის დეველოპმენტი) — მესამე
statistical მიდგომას ARIMA/SARIMA-ს გვერდით.

## რატომ Prophet?

Prophet ARIMA/SARIMA-სგან **ფუნდამენტურად განსხვავებული ფილოსოფიისაა**:

| | ARIMA/SARIMA | Prophet |
|---|---|---|
| მიდგომა | ავტორეგრესიული (წარსული მნიშვნელობები → მომავალი) | Decomposable (trend + seasonality + holidays ცალ-ცალკე მოდელირდება) |
| Stationarity | აუცილებელია (differencing) | არ არის საჭირო |
| Missing data | პრობლემურია | robust-ია |
| Holiday ეფექტები | არ აქვს ჩაშენებული მხარდაჭერა | აქვს built-in `holidays` პარამეტრი |
| სიჩქარე | ნელი (განსაკუთრებით SARIMA `m=52`-ზე) | სწრაფი — შესაძლებელია მეტ სერიაზე გაშვება |
| ინტერპრეტირებადობა | AR/MA კოეფიციენტები | ცალკეული ვიზუალური კომპონენტები (trend/seasonality/holidays) |

**კომპონენტების დაშლა (additive მოდელი):**

`y(t) = trend(t) + seasonality(t) + holidays(t) + ε(t)`

- **trend(t)** — გრძელვადიანი ზრდა/კლება, `changepoints`-ებით (ავტომატურად დეტექტირებული წერტილები სადაც trend იცვლის მიმართულებას)
- **seasonality(t)** — Fourier სერიებით მოდელირებული პერიოდული შაბლონები (yearly, weekly)
- **holidays(t)** — მომხმარებლის მიერ განსაზღვრული თარიღების ეფექტი (ჩვენს შემთხვევაში — Super Bowl, Labor Day, Thanksgiving, Christmas)

რადგან Prophet საკმაოდ სწრაფია, ამ notebook-ში მეტ store-dept წყვილზე გავუშვებთ,
ვიდრე SARIMA-ზე გავუშვით — ეს თავისთავად ერთ-ერთი პრაქტიკული უპირატესობის დემონსტრირებაცაა.


In [ ]:
!pip install prophet wandb -q


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import wandb

from prophet import Prophet


### `data_prep.py` და `evaluation.py`-ის ფუნქციები (inline)



In [ ]:
def merge_all(sales_df, features_df, stores_df):
    df = sales_df.merge(features_df, on=["Store", "Date"], how="left", suffixes=("", "_feat"))
    df = df.merge(stores_df, on="Store", how="left")

    if "IsHoliday_feat" in df.columns:
        df = df.drop(columns=["IsHoliday_feat"])

    return df


def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    is_holiday = np.asarray(is_holiday)

    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def walk_forward_splits(df, date_col="Date", n_splits=3, horizon_weeks=38):
    dates = pd.to_datetime(df[date_col])
    unique_dates = np.sort(dates.unique())

    horizon_days = horizon_weeks * 7
    splits = []
    last_date = unique_dates[-1]

    for i in range(n_splits):
        val_end = last_date - pd.Timedelta(days=horizon_days * i)
        val_start = val_end - pd.Timedelta(days=horizon_days)

        train_mask = dates < val_start
        val_mask = (dates >= val_start) & (dates <= val_end)

        if train_mask.sum() == 0:
            continue

        splits.append((train_mask.values, val_mask.values))

    return list(reversed(splits))


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()

WANDB_PROJECT = "walmart-forecasting-statistical"


## 1. მონაცემების ჩატვირთვა

იგივე Kaggle competition input path, რაც ARIMA/SARIMA notebook-ში გამოვიყენეთ.

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"

def load_raw_data_kaggle(data_dir=KAGGLE_DATA_DIR):
    train = pd.read_csv(f"{data_dir}/train.csv.zip")
    test = pd.read_csv(f"{data_dir}/test.csv.zip")
    features = pd.read_csv(f"{data_dir}/features.csv.zip")
    stores = pd.read_csv(f"{data_dir}/stores.csv")

    train["Date"] = pd.to_datetime(train["Date"])
    test["Date"] = pd.to_datetime(test["Date"])
    features["Date"] = pd.to_datetime(features["Date"])

    return train, test, features, stores


train, test, features, stores = load_raw_data_kaggle()
df = merge_all(train, features, stores)
df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

# Prophet სწრაფია, ამიტომ მეტ სერიაზე ვუშვებთ ვიდრე SARIMA-ზე (2 → 5).
# იგივე 2 სერია ARIMA/SARIMA-დანაც შენარჩუნებულია პირდაპირი შედარებისთვის + 3 დამატებული.
sample_pairs = [
    (1, 1),    # იგივე რაც ARIMA/SARIMA notebook-ში — პირდაპირი შედარებისთვის
    (20, 92),  # იგივე რაც ARIMA/SARIMA notebook-ში
    (4, 38),   # Store type A, medium volume
    (33, 5),   # Store type B/C, low volume
    (10, 72),  # Store type B, seasonal dept
]

fig, axes = plt.subplots(len(sample_pairs), 1, figsize=(12, 3 * len(sample_pairs)), sharex=True)
for ax, (store, dept) in zip(axes, sample_pairs):
    s = df[(df.Store == store) & (df.Dept == dept)].sort_values("Date")
    ax.plot(s["Date"], s["Weekly_Sales"])
    ax.set_title(f"Store {store}, Dept {dept}")
plt.tight_layout()
plt.show()


## 2. Holiday Dataframe — Prophet-ის built-in holiday მხარდაჭერა

Prophet-ს შეგვიძლია მივცეთ `holidays` dataframe (`holiday`, `ds` სვეტებით), რომელიც
ცალკე მოდელირებს ამ თარიღების ეფექტს trend/seasonality-სგან დამოუკიდებლად.

გამოვიყენებთ იმავე 4 ძირითად Walmart holiday-ს, რაც `feature_engineering.py`-ის
`add_holiday_proximity` ფუნქციაშია განსაზღვრული: Super Bowl, Labor Day, Thanksgiving, Christmas.

`lower_window`/`upper_window` საშუალებას გვაძლევს holiday ეფექტი "გავშალოთ" მიმდებარე
დღეებზეც — რადგან Walmart-ის markdown-ები ხშირად holiday-ს წინა კვირებში იწყება.

In [ ]:
holidays_df = pd.DataFrame({
    "holiday": (
        ["superbowl"] * 4 + ["labor_day"] * 4 + ["thanksgiving"] * 4 + ["christmas"] * 4
    ),
    "ds": pd.to_datetime([
        "2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08",   # Super Bowl
        "2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06",   # Labor Day
        "2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29",   # Thanksgiving
        "2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27",   # Christmas
    ]),
    "lower_window": -7,  # holiday-ს წინა კვირაც შედის ეფექტში (markdown-ების გამო)
    "upper_window": 0,
})
holidays_df.head()


## 3. Prophet fit — ერთი მაგალითი დეტალურად

პირველ რიგში ერთ სერიაზე ვხედავთ Prophet-ის output-ს დეტალურად — trend/seasonality/holiday
კომპონენტების დაშლას.

In [ ]:
def to_prophet_format(raw_df, store, dept):
    sub = raw_df[(raw_df.Store == store) & (raw_df.Dept == dept)].sort_values("Date")
    return sub.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["ds", "y"]]


example_store, example_dept = sample_pairs[0]
example_prophet_df = to_prophet_format(df, example_store, example_dept)

model = Prophet(
    holidays=holidays_df,
    yearly_seasonality=True,
    weekly_seasonality=False,   # მონაცემი უკვე კვირეულია, დღიური სეზონურობა არ არსებობს
    seasonality_mode="additive",
)
model.fit(example_prophet_df)

future = model.make_future_dataframe(periods=13, freq="W-FRI")
forecast = model.predict(future)

fig1 = model.plot(forecast)
plt.title(f"Prophet forecast — Store {example_store}, Dept {example_dept}")
plt.show()

fig2 = model.plot_components(forecast)
plt.show()


## 4. Additive vs Multiplicative Seasonality

**Additive** (`y = trend + seasonality`) გამოსადეგია, როცა სეზონური რხევის ამპლიტუდა
დროში მუდმივია. **Multiplicative** (`y = trend × seasonality`) გამოსადეგია, როცა
სეზონური რხევა იზრდება trend-თან ერთად (მაგ. თუ store-ის ზრდასთან ერთად holiday
spike-ებიც პროპორციულად იზრდება).

სწრაფი შედარება ერთ სერიაზე:

In [ ]:
def fit_prophet(prophet_df, seasonality_mode="additive", holidays=holidays_df):
    m = Prophet(
        holidays=holidays,
        yearly_seasonality=True,
        weekly_seasonality=False,
        seasonality_mode=seasonality_mode,
    )
    m.fit(prophet_df)
    return m


for mode in ["additive", "multiplicative"]:
    m = fit_prophet(example_prophet_df, seasonality_mode=mode)
    future = m.make_future_dataframe(periods=13, freq="W-FRI")
    fc = m.predict(future)
    print(f"{mode}: last forecasted value = {fc['yhat'].iloc[-1]:.2f}")


## 5. Walk-forward WMAE შეფასება ყველა სერიაზე + MLflow ლოგირება

იგივე `walk_forward_splits`/`wmae` ლოგიკა, რაც ARIMA/SARIMA notebook-შია გამოყენებული —
პირდაპირ შედარებადი შედეგებისთვის.

In [ ]:
def evaluate_prophet_walk_forward(raw_df, store, dept, seasonality_mode="additive"):
    sub = raw_df[(raw_df.Store == store) & (raw_df.Dept == dept)].sort_values("Date")
    splits = walk_forward_splits(sub, date_col="Date", n_splits=3, horizon_weeks=13)

    fold_scores = []
    for train_mask, val_mask in splits:
        train_part = sub[train_mask]
        val_part = sub[val_mask]
        if len(train_part) < 30 or len(val_part) == 0:
            continue

        prophet_train = train_part.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["ds", "y"]]

        try:
            m = fit_prophet(prophet_train, seasonality_mode=seasonality_mode)
        except Exception as e:
            print(f"Fit failed for Store {store} Dept {dept}: {e}")
            continue

        future_dates = val_part[["Date"]].rename(columns={"Date": "ds"})
        forecast = m.predict(future_dates)

        y_true = val_part["Weekly_Sales"].values
        y_pred = forecast["yhat"].values
        is_holiday = val_part["IsHoliday"].values

        score = wmae(y_true, y_pred, is_holiday)
        fold_scores.append(score)

    return np.mean(fold_scores) if fold_scores else None


prophet_results = {}
for store, dept in sample_pairs:
    run = wandb.init(project=WANDB_PROJECT, name=f"Prophet_Store{store}_Dept{dept}", reinit=True)
    wandb.config.update({
        "model_type": "Prophet", "store": store, "dept": dept,
        "seasonality_mode": "additive",
        "holidays": "superbowl,labor_day,thanksgiving,christmas",
    })

    score = evaluate_prophet_walk_forward(df, store, dept, seasonality_mode="additive")
    prophet_results[(store, dept)] = score

    if score is not None:
        wandb.log({"wmae": score})
    run.finish()

    print(f"Store {store}, Dept {dept} -> Prophet WMAE={score}")


## 6. Holidays-ის გავლენის შემოწმება (with vs without)

მოკლე abation — რამდენად აუმჯობესებს holiday dataframe-ის დამატება შედეგს, იმავე 2 სერიაზე
რაც ARIMA/SARIMA-შიც გვქონდა (`(1,1)`, `(20,92)`).

In [ ]:
def evaluate_prophet_no_holidays(raw_df, store, dept):
    sub = raw_df[(raw_df.Store == store) & (raw_df.Dept == dept)].sort_values("Date")
    splits = walk_forward_splits(sub, date_col="Date", n_splits=3, horizon_weeks=13)

    fold_scores = []
    for train_mask, val_mask in splits:
        train_part = sub[train_mask]
        val_part = sub[val_mask]
        if len(train_part) < 30 or len(val_part) == 0:
            continue

        prophet_train = train_part.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["ds", "y"]]
        m = Prophet(yearly_seasonality=True, weekly_seasonality=False)  # holidays=None
        m.fit(prophet_train)

        future_dates = val_part[["Date"]].rename(columns={"Date": "ds"})
        forecast = m.predict(future_dates)

        y_true = val_part["Weekly_Sales"].values
        y_pred = forecast["yhat"].values
        is_holiday = val_part["IsHoliday"].values

        fold_scores.append(wmae(y_true, y_pred, is_holiday))

    return np.mean(fold_scores) if fold_scores else None


ablation_pairs = sample_pairs[:2]  # იგივე (1,1) და (20,92), ARIMA/SARIMA-სთან შედარებადი
holiday_ablation = {}
for store, dept in ablation_pairs:
    run = wandb.init(project=WANDB_PROJECT, name=f"Prophet_NoHolidays_Store{store}_Dept{dept}", reinit=True)
    wandb.config.update({
        "model_type": "Prophet", "store": store, "dept": dept, "holidays": "none",
    })

    score = evaluate_prophet_no_holidays(df, store, dept)
    holiday_ablation[(store, dept)] = score

    if score is not None:
        wandb.log({"wmae": score})
    run.finish()

    print(f"Store {store}, Dept {dept} -> Prophet (no holidays) WMAE={score}")


## 7. სამივე მოდელის შედარება — ARIMA vs SARIMA vs Prophet

`ARIMA_WMAE`/`SARIMA_WMAE` მნიშვნელობები `model_experiment_ARIMA_SARIMA.ipynb`-ის
გაშვების შედეგებია.

In [ ]:
arima_wmae_manual = {(1, 1): 21595.395475189714, (20, 92): 11002.419674359162}
sarima_wmae_manual = {(1, 1): 4772.711275661171, (20, 92): 18788.33633664939}

comparison = pd.DataFrame({
    "Store": [k[0] for k in ablation_pairs],
    "Dept": [k[1] for k in ablation_pairs],
    "ARIMA_WMAE": [arima_wmae_manual.get(k) for k in ablation_pairs],
    "SARIMA_WMAE": [sarima_wmae_manual.get(k) for k in ablation_pairs],
    "Prophet_NoHolidays_WMAE": [holiday_ablation.get(k) for k in ablation_pairs],
    "Prophet_WithHolidays_WMAE": [prophet_results.get(k) for k in ablation_pairs],
})
comparison
